In [ ]:
import torch
import CT.Models.misc as misc
import CT.Inference.Kolmogorov.performance as performance
from tqdm import tqdm
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Emu_file_path = '/scratch/ql2221/Emulator_models/wandb_data/wandb/run-20251028_103612-lmys64ex/files/checkpoint_best.p'
CT_file_path = "/scratch/ql2221/CT_models/wandb_data/wandb/run-20251213_225527-d0htlgni/files/checkpoint_last.p"

CT = misc.load_diffusion_model(CT_file_path).to(device)
Emu = misc.load_model(Emu_file_path).to(device)

In [1]:
data_dict = torch.load("/scratch/ql2221/PDE_data/Kolmogorov_flow/Reynolds10k/Long_numerical_rollout.p")

NameError: name 'torch' is not defined

In [3]:
data = data_dict["data"]
x = data/4.44
x = x.to(device)
del data, data_dict

In [4]:
emu_rollout = performance.run_emu(x[:,0:1],emu = Emu, n_steps=20000,silent=False,sigma=None)

100%|██████████| 19999/19999 [01:48<00:00, 184.36it/s]


In [5]:
Therm_rollout= performance.roll_and_anchor(ics = x[:,0:1], emu = Emu, CT = CT, n_steps=40000, short_lag_int = 1, freq = 5, lag_ratio = 5, s_init = 8, max_long_lag = 100, starting_time = 10000, silence = True, device = x.device)

tensor([ 5,  7,  9, 11, 11], device='cuda:0')
tensor([0, 0, 0, 1, 1], device='cuda:0')
tensor([0, 0, 0, 1, 1], device='cuda:0')
tensor([0, 0, 0, 1, 1], device='cuda:0')
tensor([0, 0, 0, 1, 1], device='cuda:0')
tensor([0, 1, 0, 1, 1], device='cuda:0')
tensor([0, 1, 1, 1, 1], device='cuda:0')
tensor([0, 1, 1, 3, 1], device='cuda:0')
tensor([0, 3, 1, 3, 1], device='cuda:0')
tensor([0, 3, 3, 5, 1], device='cuda:0')
tensor([0, 3, 5, 5, 1], device='cuda:0')
tensor([0, 3, 5, 5, 1], device='cuda:0')
tensor([0, 3, 5, 5, 1], device='cuda:0')
tensor([0, 3, 5, 5, 1], device='cuda:0')
tensor([0, 3, 5, 5, 1], device='cuda:0')
tensor([0, 3, 3, 5, 1], device='cuda:0')
tensor([0, 5, 5, 5, 3], device='cuda:0')
tensor([0, 5, 5, 5, 3], device='cuda:0')
tensor([0, 5, 5, 5, 3], device='cuda:0')
tensor([0, 5, 5, 5, 3], device='cuda:0')
tensor([0, 5, 5, 5, 3], device='cuda:0')
tensor([0, 5, 5, 7, 3], device='cuda:0')
tensor([1, 5, 5, 9, 3], device='cuda:0')
tensor([0, 0, 0, 2, 0], device='cuda:0')
tensor([0, 

In [12]:
import os
import matplotlib
matplotlib.use("Agg")  # MUST be before importing pyplot

import matplotlib.pyplot as plt
from matplotlib import animation
from tqdm import tqdm


def make_movie(
    state_vector,
    save_path="emulator_movie.mp4",
    fps=30,
    vmin=None,
    vmax=None,
    stride=10,
    dpi=100,
    figsize_per_panel=4,
    fallback_to_gif=True,
):
    """
    Render a movie from a tensor/array shaped (B, T, H, W).

    - If save_path ends with .mp4, uses FFmpeg if available.
    - If FFmpeg is not available and fallback_to_gif=True, saves a .gif instead.
    - If save_path ends with .gif, uses PillowWriter (no FFmpeg required).
    """

    # Convert to numpy safely (works for torch tensors)
    try:
        data = state_vector.detach().cpu().numpy()
    except AttributeError:
        data = state_vector

    if data.ndim != 4:
        raise ValueError(f"Expected shape (B,T,H,W); got {data.shape}")

    B, T, H, W = data.shape
    frame_indices = list(range(0, T, stride))
    n_frames = len(frame_indices)
    if n_frames == 0:
        raise ValueError(f"No frames to render: T={T}, stride={stride} produced empty frame list.")

    # Figure / axes
    if B == 1:
        fig, ax = plt.subplots(
            figsize=(figsize_per_panel, figsize_per_panel),
            constrained_layout=True
        )
        axes = [ax]
    else:
        fig, axes = plt.subplots(
            1, B,
            figsize=(figsize_per_panel * B, figsize_per_panel),
            constrained_layout=True
        )
        axes = list(axes)

    # Initial frame
    ims = []
    t0 = frame_indices[0]
    for b in range(B):
        ax = axes[b]
        im = ax.imshow(data[b, t0], cmap="viridis", vmin=vmin, vmax=vmax)
        ax.set_title(f"Batch {b}, Timestep {t0}")
        ax.set_xticks([])
        ax.set_yticks([])
        ims.append(im)

    frames_iter = tqdm(range(n_frames), desc="Building frames", total=n_frames)

    def update(i):
        t = frame_indices[i]
        for b in range(B):
            ims[b].set_data(data[b, t])
            axes[b].set_title(f"Batch {b}, Timestep {t}")
        return ims

    ani = animation.FuncAnimation(fig, update, frames=frames_iter, blit=False, repeat=False)

    lower = save_path.lower()
    out_path = save_path

    try:
        if lower.endswith(".gif"):
            writer = animation.PillowWriter(fps=fps)
            ani.save(save_path, writer=writer, dpi=dpi)

        elif lower.endswith(".mp4"):
            ffmpeg_ok = animation.writers.is_available("ffmpeg")

            if not ffmpeg_ok:
                if not fallback_to_gif:
                    raise RuntimeError(
                        "FFmpeg writer is not available, and fallback_to_gif=False. "
                        "Install ffmpeg or set save_path to end with .gif."
                    )

                # Fallback: write GIF next to requested mp4
                root, _ = os.path.splitext(save_path)
                out_path = root + ".gif"
                writer = animation.PillowWriter(fps=fps)
                ani.save(out_path, writer=writer, dpi=dpi)
                print(f"FFmpeg not available; saved GIF instead: {out_path}")
            else:
                # If ffmpeg exists, still use a robust codec choice
                writer = animation.FFMpegWriter(
                    fps=fps,
                    codec="mpeg4",
                    bitrate=2000,
                    extra_args=["-pix_fmt", "yuv420p"],
                )
                ani.save(save_path, writer=writer, dpi=dpi)

        else:
            raise ValueError("save_path must end with .mp4 or .gif")

    finally:
        plt.close(fig)

    print("movie saved:", out_path)
    return out_path


In [ ]:
make_movie(Therm_rollout[:,::10], save_path="therm_rollout.mp4", fps=30)


Building frames:   0%|          | 0/400 [00:00<?, ?it/s]
Exception ignored in: 'read_from_file_callback'
Traceback (most recent call last):
  File "/ext3/miniforge3/lib/python3.12/site-packages/matplotlib/backends/backend_agg.py", line 219, in get_text_width_height_descent
OSError: [Errno 107] Transport endpoint is not connected
Exception ignored in: 'read_from_file_callback'
Traceback (most recent call last):
  File "/ext3/miniforge3/lib/python3.12/site-packages/matplotlib/backends/backend_agg.py", line 219, in get_text_width_height_descent
OSError: [Errno 107] Transport endpoint is not connected
Exception ignored in: 'read_from_file_callback'
Traceback (most recent call last):
  File "/ext3/miniforge3/lib/python3.12/site-packages/matplotlib/backends/backend_agg.py", line 219, in get_text_width_height_descent
OSError: [Errno 107] Transport endpoint is not connected
Exception ignored in: 'read_from_file_callback'
Traceback (most recent call last):
  File "/ext3/miniforge3/lib/python3.

In [12]:
x = x.to(device)
empty = torch.zeros(4,emu_rollout.shape[1],emu_rollout.shape[2],emu_rollout.shape[3]).to(device)
tensor = torch.cat((x[:,:emu_rollout.shape[1]], emu_rollout, Therm_rollout), dim = 0)
titles = [ "GT", "non", "non", "non", "non", "Emu", "non", "non", "non", "non", "Therm", "non", "non", "non", "non"]

In [13]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from tqdm import tqdm

def make_movie(
    state_vector,
    save_path="emulator_movie.mp4",
    fps=30,
    vmin=None,
    vmax=None,
    stride=10,
    titles=None,
    cmap="viridis",
    interpolation="nearest",
    dpi=150,
    bitrate=1800,
):
    """
    Create a 3x5 grid movie from a 4D tensor/array of shape (B, T, H, W), using only every `stride`-th frame.
    - B must be 15 (3 rows x 5 columns).
    - One shared colorbar per row (fixed over time).
    - Optional custom title per panel via `titles` (len 15, row-major).
    - tqdm progress bar while writing with ffmpeg.

    Parameters
    ----------
    state_vector : torch.Tensor | np.ndarray of shape (B, T, H, W)
    save_path    : str, output video file path (e.g., "emulator_movie.mp4")
    fps          : int, frames per second
    vmin, vmax   : float or None
        If provided, used as a global color scale across ALL rows.
        If None, each row gets auto-computed (fixed) limits over INCLUDED frames.
    stride       : int, include every `stride`-th frame
    titles       : list[str] or None, length 15 (row-major: r0 c0..4, r1 c0..4, r2 c0..4)
    cmap         : str, matplotlib colormap name
    interpolation: str, imshow interpolation (e.g., "nearest", "bilinear", "bicubic")
    dpi          : int, figure DPI for saved video
    bitrate      : int, bitrate for FFMpegWriter
    """

    # Convert to numpy without requiring torch as a dependency at import time
    if hasattr(state_vector, "detach"):  # likely a torch.Tensor
        arr = state_vector.detach().cpu().numpy()
    else:
        arr = np.asarray(state_vector)

    if arr.ndim != 4:
        raise ValueError(f"Expected [B, T, H, W], got {arr.shape}")
    B, T, H, W = arr.shape
    if B != 15:
        raise ValueError(f"B must be 15 for a 3x5 grid, got B={B}")
    if T < 1:
        raise ValueError("T must be >= 1")

    # Frame indices to use
    stride = max(1, int(stride))
    frame_indices = list(range(0, T, stride))
    if frame_indices[-1] != T - 1:
        frame_indices.append(T - 1)  # ensure last frame is included

    # Helper for row/col -> batch index
    def bidx(r, c):
        return r * 5 + c

    # Determine color limits
    row_limits = []
    if vmin is not None or vmax is not None:
        # Global scale across all rows
        gmin, gmax = vmin, vmax
        if gmin is None or gmax is None:
            used = arr[:, frame_indices, :, :]
            amin = float(np.nanmin(used))
            amax = float(np.nanmax(used))
            if gmin is None:
                gmin = amin
            if gmax is None:
                gmax = amax
        row_limits = [(gmin, gmax)] * 3
    else:
        # Per-row fixed scale computed from included frames only
        for r in range(3):
            row_data = arr[bidx(r, 0):bidx(r, 0)+5, :, :, :]  # [5, T, H, W]
            row_used = row_data[:, frame_indices, :, :]
            rmin = float(np.nanmin(row_used))
            rmax = float(np.nanmax(row_used))
            row_limits.append((rmin, rmax))

    # Titles
    if titles is None:
        titles = [f"Panel {i+1}" for i in range(15)]
    if len(titles) != 15:
        raise ValueError("`titles` must have length 15 (row-major order).")

    # Figure and axes
    fig, axes = plt.subplots(3, 5, figsize=(15, 9), dpi=dpi, squeeze=False)

    # Create initial images
    ims = []  # keep references for animation update
    for r in range(3):
        vmin_r, vmax_r = row_limits[r]
        for c in range(5):
            ax = axes[r, c]
            im = ax.imshow(
                arr[bidx(r, c), frame_indices[0]],
                cmap=cmap,
                vmin=vmin_r,
                vmax=vmax_r,
                interpolation=interpolation,
                origin="upper",
                animated=True,
            )
            ax.set_xticks([])
            ax.set_yticks([])
            ax.set_title(titles[bidx(r, c)], fontsize=10)
            ims.append(im)

        # One colorbar per row, shared across the five axes in that row
        cbar = fig.colorbar(
            ims[r*5 + 4],  # any image from this row
            ax=list(axes[r, :]),
            orientation="vertical",
            fraction=0.046,
            pad=0.02
        )
        cbar.set_label(
            f"Row {r+1} scale [{row_limits[r][0]:.3g}, {row_limits[r][1]:.3g}]",
            fontsize=9
        )

    # Global suptitle shows the current frame index
    suptitle = fig.suptitle(f"Timestep {frame_indices[0]}", fontsize=12)

    # Update function used by both preview animation and manual writer loop
    def update(i):
        f = frame_indices[i]
        for r in range(3):
            for c in range(5):
                ims[r*5 + c].set_data(arr[bidx(r, c), f])
        suptitle.set_text(f"Timestep {f}")
        return ims

    # Build the animation object (useful if you want to preview in notebooks),
    # but we'll save manually to expose tqdm progress.
    _ = animation.FuncAnimation(
        fig, update, frames=len(frame_indices), blit=False, repeat=False
    )

    # Save with a manual loop + tqdm progress bar
    writer = animation.FFMpegWriter(fps=fps, bitrate=bitrate)
    with writer.saving(fig, save_path, dpi=dpi):
        for i in tqdm(range(len(frame_indices)), desc="Rendering movie"):
            update(i)
            writer.grab_frame()

    plt.close(fig)
    print(f"🎬 Movie saved: {save_path}")


In [14]:
make_movie(
    state_vector = tensor,
    save_path="compare_movie.mp4",
    fps=30,
    vmin=-4,
    vmax=4,
    stride=10,
    titles=titles,
    cmap="viridis",
    interpolation="nearest",
    dpi=150,
    bitrate=1800,
)

Rendering movie: 100%|██████████| 3001/3001 [06:57<00:00,  7.19it/s]


🎬 Movie saved: compare_movie.mp4


/ext3/miniforge3/lib/python3.12/site-packages/matplotlib/animation.py:908: UserWarning: Animation was deleted without rendering anything. This is most likely not intended. To prevent deletion, assign the Animation to a variable, e.g. `anim`, that exists until you output the Animation using `plt.show()` or `anim.save()`.
  warnings.warn(


In [15]:
tensor.shape

torch.Size([15, 30000, 64, 64])

In [16]:
tensor[5:10] = 0

In [17]:
make_movie(
    state_vector = tensor,
    save_path="compare_movie.mp4",
    fps=30,
    vmin=-4,
    vmax=4,
    stride=10,
    titles=titles,
    cmap="viridis",
    interpolation="nearest",
    dpi=150,
    bitrate=1800,
)

Rendering movie: 100%|██████████| 3001/3001 [06:43<00:00,  7.44it/s]


🎬 Movie saved: compare_movie.mp4


/ext3/miniforge3/lib/python3.12/site-packages/matplotlib/animation.py:908: UserWarning: Animation was deleted without rendering anything. This is most likely not intended. To prevent deletion, assign the Animation to a variable, e.g. `anim`, that exists until you output the Animation using `plt.show()` or `anim.save()`.
  warnings.warn(
